# Mid-Level API — Overview

This notebook introduces the building blocks of the mid-level PGM API:

1. **Variables** — declarative descriptions of nodes in a DAG.
2. **`ParametricCPD`** — neural-network-parameterised conditional distributions.
3. **`ProbabilisticModel`** — the directed factor graph wrapper.
4. **Inference engines** — `DeterministicInference`, `AncestralInference`, `VariationalInference`.

We use a tiny synthetic XOR-style concept-bottleneck problem so every step fits on screen. The applied counterpart is `mid_level_api_asia.ipynb`.

In [21]:
import warnings
import torch
import torch.nn.functional as F
import torch.nn as nn
import pyro
import pyro.distributions as dist

from torch_concepts.nn.modules.pgm import (
    ConceptVariable, ExogenousVariable,
    ParametricCPD, ProbabilisticModel,
    DeterministicInference, AncestralInference, VariationalInference,
)

warnings.filterwarnings('ignore')
torch.manual_seed(0); pyro.set_rng_seed(0)

## 1. Variables

A `Variable` is a **pure spec object** — it does not hold parameters and does not participate in autograd. Two concrete subclasses cover every node:

| Class | Role |
|---|---|
| `ConceptVariable` | An interpretable concept (may be supervised, latent, or deterministic via `dist.Delta`) |
| `ExogenousVariable` | A non-interpretable input (e.g. an embedding `x`); always a root, always observed |

Whether a variable is *latent* at inference time is decided by the engine: it is simply any variable not present in `data` at the call site. The model itself carries no observability metadata.

In [2]:
x = ExogenousVariable('x', size=16)
c1, c2 = ConceptVariable(['c1', 'c2'], distribution=dist.Bernoulli)
y = ConceptVariable('y', distribution=dist.OneHotCategorical, size=2)

for v in (x, c1, c2, y):
    print(v)

ExogenousVariable(concept='x', distribution=None, size=16)
ConceptVariable(concept='c1', distribution=Bernoulli, size=1)
ConceptVariable(concept='c2', distribution=Bernoulli, size=1)
ConceptVariable(concept='y', distribution=OneHotCategorical, size=2)


## 2. `ParametricCPD`

Each variable owns one CPD whose neural network produces the distribution's **parameters** (probabilities for discrete families, location/scale for continuous ones). `_split_params` applies the appropriate bounding activation:

| Distribution | Output keys | Bounding |
|---|---|---|
| `Bernoulli` | `probs` | sigmoid |
| `OneHotCategorical`, `Categorical` | `probs` | softmax |
| `Normal` | `loc`, `scale` | identity / softplus |
| `MultivariateNormal` | `loc`, `scale_tril` | identity / lower-tri w/ softplus diag |
| `Delta` | `v` | identity (sampling = identity on NN output) |

Roots are emitted via `pyro.deterministic` and their parametrisation is mandatory (`nn.Identity()` for an untransformed input).

In [42]:
cpd_x  = ParametricCPD('x', nn.Identity())
cpd_c1 = ParametricCPD('c1', nn.Linear(16, 1), parents=[x])
cpd_c2 = ParametricCPD('c2', nn.Linear(16, 1), parents=[x])
# y reads c1, c2 (size 1 each) -> input dim 2; OneHotCategorical(size=2) needs 2 outputs.
cpd_y  = ParametricCPD('y', nn.Sequential(nn.Linear(2,2), nn.ReLU(), nn.Linear(2,2)), parents=[c1, c2])

for cpd in (cpd_x, cpd_c1, cpd_c2, cpd_y):
    print(cpd.concept, cpd.parametrization)

x Identity()
c1 Linear(in_features=16, out_features=1, bias=True)
c2 Linear(in_features=16, out_features=1, bias=True)
y Sequential(
  (0): Linear(in_features=2, out_features=2, bias=True)
  (1): ReLU()
  (2): Linear(in_features=2, out_features=2, bias=True)
)


## 3. `ProbabilisticModel`

The PGM ties variables and factors together, resolves parents, runs a topological sort, and exposes itself as a Pyro stochastic function `pgm(data)` that emits one `pyro.sample` site per variable (with `obs=data[name]` for observed variables) and `pyro.deterministic` for roots.

In [43]:
pgm = ProbabilisticModel(
    variables=[x, c1, c2, y],
    factors=[cpd_x, cpd_c1, cpd_c2, cpd_y],
)
[v.concept for v in pgm.sorted_variables]

['x', 'c1', 'c2', 'y']

## 4. A tiny synthetic XOR-style dataset

In [31]:
def make_batch(B=256):
    X = torch.randn(B, 16)
    # Two ground-truth concepts read off two coordinates of x.
    C1 = (X[:, 0] > 0).float()
    C2 = (X[:, 1] > 0).float()
    Y_int = (C1.long() ^ C2.long())
    Y = torch.nn.functional.one_hot(Y_int, num_classes=2).float()
    return {'x': X, 'c1': C1, 'c2': C2, 'y': Y}

batch = make_batch()
{k: v.shape for k, v in batch.items()}

{'x': torch.Size([256, 16]),
 'c1': torch.Size([256]),
 'c2': torch.Size([256]),
 'y': torch.Size([256, 2])}

## 5. `DeterministicInference` — fully supervised CBM training

`DeterministicInference` propagates the NN-derived **parameters** (probabilities / means) to children — no sampling. The `p_int` argument controls per-sample-per-variable teacher-forcing: with `p_int=1.0` (default), whenever a label is present in `data` for a non-evidence variable it is teacher-forced; with `p_int=0.0` only NN outputs flow.

In [47]:
det = DeterministicInference(pgm, p_int=1.0)
optim = torch.optim.Adam(pgm.parameters(), lr=5e-2)
loss_fn = F.binary_cross_entropy

for step in range(10000):
    batch = make_batch()
    out = det(query=['c1', 'c2', 'y'], evidence=['x'], data=batch)
    
    loss_c1 = loss_fn(out.model_params['c1']['probs'], batch['c1'])
    loss_c2 = loss_fn(out.model_params['c2']['probs'], batch['c2'])
    loss_y = loss_fn(out.model_params['y']['probs'], batch['y'])
    loss = loss_c1 + loss_c2 + loss_y
    
    optim.zero_grad()
    loss.backward()
    optim.step()

print('final det loss:', float(loss))

final det loss: 0.6972878575325012


### Test-time prediction

At test time only `x` is observed. With `p_int=0.0` the engine never teacher-forces, so all parameters are pure NN outputs.

In [48]:
test_batch = make_batch(B=512)
det_test = DeterministicInference(pgm, p_int=0.0)
with torch.no_grad():
    out = det_test(query=['c1', 'c2', 'y'], evidence=['x'], data={'x': test_batch['x']})
y_pred = out.model_params['y']['probs'].argmax(dim=-1)
y_true = test_batch['y'].argmax(dim=-1)
acc = (y_pred == y_true).float().mean()
print(f'test accuracy on y: {acc.item():.3f}')

test accuracy on y: 0.492


## 6. `AncestralInference` — single-sample propagation

`AncestralInference` draws one (reparameterised / Straight-Through-relaxed) sample per variable. `model_params[v]` carries the NN-derived prior parameters at the sampled trace; `samples[v]` carries the actual sampled value. There is no $S$ dimension.

In [8]:
anc = AncestralInference(pgm, p_int=0.0, initial_temperature=1.0, annealing='constant')
with torch.no_grad():
    out = anc(query=['c1', 'c2', 'y'], evidence=['x'], data={'x': test_batch['x']})
print('samples shapes:', {k: tuple(v.shape) for k, v in out.samples.items()})
print('model_params:', list(out.model_params.keys()))

samples shapes: {'x': (512, 16), 'c1': (512,), 'c2': (512,), 'y': (512, 2)}
model_params: ['c1', 'c2', 'y']


## 7. `VariationalInference` — latent concepts via amortised guides

Latent variables are declared at engine-construction time via `latents=[...]`. The engine eagerly builds one guide per latent using a family-keyed `default_guides` dict; the user can override per family.

Below we treat `c2` as a hidden concept (we drop it from `data`) and let the guide infer it from `x`.

In [9]:
# Fresh PGM so we don't reuse the deterministic-trained weights.
x2 = ExogenousVariable('x', size=16)
c1b, c2b = ConceptVariable(['c1', 'c2'], distribution=dist.Bernoulli)
yb = ConceptVariable('y', distribution=dist.OneHotCategorical, size=2)
pgm2 = ProbabilisticModel(
    variables=[x2, c1b, c2b, yb],
    factors=[
        ParametricCPD('x', nn.Identity()),
        ParametricCPD('c1', nn.Linear(16, 1), parents=[x2]),
        ParametricCPD('c2', nn.Linear(16, 1), parents=[x2]),
        ParametricCPD('y', nn.Linear(2, 2), parents=[c1b, c2b]),
    ],
)

vi = VariationalInference(pgm2, latents=['c2'], initial_temperature=1.0, annealing='constant')
optim = torch.optim.Adam(list(pgm2.parameters()) + list(vi.parameters()), lr=5e-2)

for step in range(200):
    batch = make_batch()
    # c2 is hidden: drop it from data.
    data_no_c2 = {k: v for k, v in batch.items() if k != 'c2'}
    out = vi(query=['c1', 'c2', 'y'], evidence=['x', 'c1', 'y'], data=data_no_c2)
    # Data fit on observed concepts only.
    data_fit = (
        dist.Bernoulli(probs=out.model_params['c1']['probs']).log_prob(batch['c1']).mean()
        + dist.OneHotCategorical(probs=out.model_params['y']['probs']).log_prob(batch['y']).mean()
    )
    # MC KL for c2 (analytic KL not registered for the relaxed family).
    q_kw = out.guide_params['c2']
    p_probs = out.model_params['c2']['probs']
    q_dist = dist.RelaxedBernoulliStraightThrough(**q_kw)
    z = q_dist.rsample()
    p_dist = dist.Bernoulli(probs=p_probs)
    kl = (q_dist.log_prob(z) - p_dist.log_prob(z)).mean()
    loss = -(data_fit - kl)
    optim.zero_grad(); loss.backward(); optim.step()

print('final VI loss:', float(loss))

final VI loss: 1.5221360921859741


### Posterior evaluation

Use the variational guide to predict `c2` from `x` on test data.

In [10]:
with torch.no_grad():
    test_batch = make_batch(B=512)
    out = vi(query=['c2', 'y'], evidence=['x', 'c1', 'y'],
             data={'x': test_batch['x'], 'c1': test_batch['c1'], 'y': test_batch['y']})
    c2_q_probs = out.guide_params['c2']['probs']
    c2_pred = (c2_q_probs > 0.5).float()
    acc = (c2_pred == test_batch['c2']).float().mean()
print(f'guide-posterior accuracy on hidden c2: {acc.item():.3f}')

guide-posterior accuracy on hidden c2: 0.432


## 8. Pyro interoperability — `Trace_ELBO` directly on `pgm`

Because `pgm.forward(data)` is a plain Pyro callable and `vi.guide_fn` is a plain `PyroModule`-backed callable, we can hand them to Pyro's built-in `Trace_ELBO` to compute the loss without using the engine's `_run`.

In [11]:
from pyro.infer import Trace_ELBO
elbo = Trace_ELBO()

batch = make_batch()
obs = {k: batch[k] for k in ['x', 'c1', 'y']}  # evidence (c2 is latent)
B = obs['x'].shape[0]

def model_fn(data):
    with pyro.plate('data', B):
        return pgm2(data)

def guide_fn(data):
    with pyro.plate('data', B):
        return vi.guide(data, vi.temperature)

loss = elbo.differentiable_loss(model_fn, guide_fn, obs)
print('Trace_ELBO loss:', float(loss))

Trace_ELBO loss: 198.9975128173828
